# V7 — **Một lần Run All** cho lô 003–021 (944 positive TEST)

Đã hoàn thành **lô 000, 001, 002** (150 cặp positive mới). Notebook này tự chạy các lô **003 đến 021** theo đúng V7 đã sử dụng: CountGD++ → V4 → V5 → V6 → V7.

**Hướng dẫn nhanh:** Add Input `hihihi5` (FSC-147) + `chaylai3` (batch manifests 22 lô), bật **GPU + Internet**, nhấn **Run All** một lần. Không cần sửa `BATCH_INDEX`. Nếu lần trước đã xuất thêm checkpoint, Add Input các file checkpoint để notebook tự bỏ qua ảnh đã xử lý.

⚠️ **Giới hạn Kaggle:** một lần bấm Run All có thể chạy tự động nhiều lô, nhưng không có gì đảm bảo 944 ảnh sẽ hoàn tất trong một phiên GPU; hết phiên/quota vẫn phải khôi phục checkpoint và chạy lại. Notebook xuất ZIP sau mỗi stage/lô, và ghi ZIP tạm định kỳ khi đang xử lý. Khuyến nghị **Save Version** hoặc tải checkpoint để giữ kết quả sang phiên mới.

**Không thay đổi quy tắc suy luận V7** hay prompt so với notebook gốc; chỉ tái dùng model trong từng chunk để giảm khởi tạo lặp. V7 chưa được kiểm thử CUDA ở môi trường tạo notebook; nên quan sát một chunk đầu.


In [ ]:
# CELL 1 — CONFIG
from pathlib import Path
import random, json, os, sys, zipfile, time, gc, re
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
RUN_MODE = "FULL"   # Luôn FULL khi chạy tự động
SMOKE_N = 2
BATCH_INDEX = 3        # Lô 000, 001, 002 đã có checkpoint
BATCH_SIZE = 50
COUNTGD_THRESH = 0.23
CONTACT_SHEET_SIZE = 12
PANEL_SIZE = 224
CROP_PADDING_FRAC = 0.30
MAX_POS_BOXES_TO_VERIFY_PER_IMAGE = 80
DISCOVERY_REJECTED_CROPS = 6
MAX_KEEP_CROPS = 6
MAX_REJECT_CROPS = 6
MIN_KEEP_BOXES_FOR_TARGET_CONSENSUS = 5
RETENTION_THRESHOLD = 0.50
QWEN_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
COUNTGD_GITHUB = "https://github.com/niki-amini-naieni/CountGDPlusPlus.git"
COUNTGD_REPO = Path("/kaggle/working/CountGDPlusPlus")
COUNTGD_GDRIVE_ID = "1j6N22TtKu2NVcKpgfrf-sJHGeLDqs9hs"
KEEP_ON_PARSE_FAILURE = True

OUTPUT_DIR = Path(f"/kaggle/working/v7_official_test_batch_{BATCH_INDEX:03d}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATES_CSV = OUTPUT_DIR/"stage_a_countgd_candidates.csv"
V4_PRED_CSV = OUTPUT_DIR/"stage_b_v4_box_predictions.csv"
V4_IMAGE_CSV = OUTPUT_DIR/"stage_b_v4_image_counts.csv"
DISCOVERY_CSV = OUTPUT_DIR/"stage_c_v5_confounder_discovery.csv"
FEEDBACK_CSV = OUTPUT_DIR/"stage_d_v5_feedback.csv"
ROUTER_CSV = OUTPUT_DIR/"stage_e_v6_router.csv"
FINAL_CSV = OUTPUT_DIR/f"v7_shard_predictions_{BATCH_INDEX:03d}.csv"
AUDIT_CSV = OUTPUT_DIR/f"v7_shard_audit_{BATCH_INDEX:03d}.csv"
print("RUN_MODE",RUN_MODE,"BATCH_INDEX",BATCH_INDEX,"OUTPUT_DIR",OUTPUT_DIR)

# Chỉ cần Run All. Tự chạy từ 003 đến 021 (tổng 944 ảnh).
START_BATCH = 3
END_BATCH = 21
CHUNK_BATCHES = 2    # 2 lô/lượt: giảm tải model nhưng có kết quả đầy đủ sớm
AUTO_ZIP_EVERY_WRITES = 12   # 12 cập nhật CSV => tạo ZIP tạm đề phòng ngắt phiên
assert RUN_MODE == "FULL" and BATCH_SIZE == 50
assert 0 <= START_BATCH <= END_BATCH <= 21
assert CHUNK_BATCHES >= 1
print(f"AUTO RUN: batches {START_BATCH:03d}..{END_BATCH:03d}")


In [ ]:
ORIGINAL_INPUT_CELL = r'''
# CELL 2 — Input discovery + exact pending TEST manifests
from pathlib import Path
import zipfile, io
ROOTS=[Path('/kaggle/input'),Path('/kaggle/working')]
def find_input(name):
    hits=[]
    for root in ROOTS:
        if root.exists(): hits += list(root.rglob(name))
    hits=[p for p in hits if p.is_file()]
    return sorted(hits,key=lambda p:(0 if str(p).startswith('/kaggle/input') else 1,len(str(p))))[0] if hits else None

def find_image_dir():
    hits=[]
    for root in ROOTS:
        if root.exists():hits += [p for p in root.rglob('images_384_VarV2') if p.is_dir()]
    return max(hits,key=lambda p:len(list(p.glob('*.jpg')))) if hits else None

IMAGE_DIR=find_image_dir()
if IMAGE_DIR is None:raise FileNotFoundError('Thiếu FSC147 images_384_VarV2 trong Add Input')
image_index={p.name:str(p) for p in IMAGE_DIR.glob('*.jpg')}
progress=find_input('v7_next_official_test_progress.zip')
joint=find_input('nckh_v7_fsc147_111_joint_export.zip')
# Kaggle often exposes uploaded ZIP files as extracted datasets instead of raw ZIP.
extracted_batch=find_input(f'v7_test_batch_{BATCH_INDEX:03d}_of_022.csv')
extracted_pending=find_input('v7_pending_predictions.csv')
if progress:
    with zipfile.ZipFile(progress) as z:
        members=sorted([n for n in z.namelist() if re.search(r'v7_test_batch_\d{3}_of_022\.csv$',n)])
        if len(members)!=22:raise ValueError(f'Progress zip cần 22 batch manifests, thấy {len(members)}')
        if not 0<=BATCH_INDEX<len(members):raise ValueError('BATCH_INDEX phải 0..21')
        if BATCH_SIZE!=50:raise ValueError('Với progress zip, BATCH_SIZE phải là 50')
        batch_df=pd.read_csv(io.BytesIO(z.read(members[BATCH_INDEX])))
    print('Using progress ZIP:',members[BATCH_INDEX])
elif extracted_batch is not None:
    if BATCH_SIZE!=50:raise ValueError('BATCH_SIZE phải là 50 khi dùng extracted batch manifests')
    batch_df=pd.read_csv(extracted_batch)
    print('Using extracted batch CSV:',extracted_batch)
elif joint:
    with zipfile.ZipFile(joint) as z:
        pending=pd.read_csv(io.BytesIO(z.read('v7_pending_predictions.csv')))
    pending=pending[pending['split'].astype(str).eq('test') & pending['pair_type'].astype(str).eq('POSITIVE')]
    pending=pending.sort_values('group_id').reset_index(drop=True)
    batch_df=pending.iloc[BATCH_INDEX*BATCH_SIZE:(BATCH_INDEX+1)*BATCH_SIZE].copy()
    print('Using joint ZIP pending TEST manifest')
elif extracted_pending is not None:
    pending=pd.read_csv(extracted_pending)
    pending=pending[pending['split'].astype(str).eq('test') & pending['pair_type'].astype(str).eq('POSITIVE')]
    pending=pending.sort_values('group_id').reset_index(drop=True)
    batch_df=pending.iloc[BATCH_INDEX*BATCH_SIZE:(BATCH_INDEX+1)*BATCH_SIZE].copy()
    print('Using extracted joint pending TEST manifest')
else:
    raise FileNotFoundError('Add Input progress/joint ZIP hoặc thư mục đã giải nén chứa batch manifests/pending CSV')

if len(batch_df)==0:raise ValueError('Batch rỗng')
assert batch_df['group_id'].is_unique
assert batch_df['pair_type'].astype(str).eq('POSITIVE').all()
assert batch_df['split'].astype(str).eq('test').all()
assert (pd.to_numeric(batch_df['gt_count'])>0).all()
# Kaggle mount paths có thể thay đổi giữa phiên: KHÔNG dùng image_path lưu trong CSV cũ.
batch_df['image_path']=batch_df['image_name'].astype(str).map(image_index)
if batch_df['image_path'].isna().any():
    raise FileNotFoundError('Thiếu ảnh: '+str(batch_df.loc[batch_df.image_path.isna(),'image_name'].tolist()))
if RUN_MODE=='SMOKE':batch_df=batch_df.head(SMOKE_N).copy()
batch_df=batch_df.reset_index(drop=True)
batch_df.to_csv(OUTPUT_DIR/'batch_manifest_active.csv',index=False)
print('N images',len(batch_df),'first:',batch_df[['group_id','image_name','target']].head().to_dict('records'))

'''
ORIGINAL_RESTORE_CELL = r'''
# CELL 3 — Restore optional checkpoint from exported earlier run of SAME BATCH.
# This cell is conservative: it does not overwrite existing outputs.
oldzip=find_input(f'v7_official_test_batch_{BATCH_INDEX:03d}_checkpoint.zip')
if oldzip:
    with zipfile.ZipFile(oldzip) as z:
        restored=[]
        for name in z.namelist():
            base=Path(name).name
            if not base.endswith('.csv'):continue
            dest=OUTPUT_DIR/base
            if not dest.exists():
                dest.write_bytes(z.read(name));restored.append(base)
        print('Restored checkpoint:',restored)
else: print('No saved checkpoint ZIP for this batch (normal on first run)')

def atomic_csv(df,path):
    tmp=Path(str(path)+'.tmp')
    df.to_csv(tmp,index=False);tmp.replace(path)

def read_checkpoint(path):
    return pd.read_csv(path) if Path(path).exists() and Path(path).stat().st_size else pd.DataFrame()

def completed(df,key='group_id'):
    if df.empty:return set()
    if 'status' in df.columns:
        return set(df.loc[df.status.astype(str).eq('ok'),key].astype(str))
    return set(df[key].astype(str))

print('Ready')

'''
# Đọc manifest gốc cho TỪNG lô và restore checkpoint nếu có trong Input.
# Sử dụng Cell 2+3 của notebook gốc, không tái chia train/val/test.
import shutil, traceback, datetime
from IPython.display import display, FileLink

_manifest_by_batch={}

def _paths_for_batch(bi):
    root=Path(f'/kaggle/working/v7_official_test_batch_{bi:03d}')
    root.mkdir(parents=True,exist_ok=True)
    return {
      'BATCH_INDEX':bi,'OUTPUT_DIR':root,
      'CANDIDATES_CSV':root/'stage_a_countgd_candidates.csv',
      'V4_PRED_CSV':root/'stage_b_v4_box_predictions.csv',
      'V4_IMAGE_CSV':root/'stage_b_v4_image_counts.csv',
      'DISCOVERY_CSV':root/'stage_c_v5_confounder_discovery.csv',
      'FEEDBACK_CSV':root/'stage_d_v5_feedback.csv',
      'ROUTER_CSV':root/'stage_e_v6_router.csv',
      'FINAL_CSV':root/f'v7_shard_predictions_{bi:03d}.csv',
      'AUDIT_CSV':root/f'v7_shard_audit_{bi:03d}.csv',
    }

for _bi in range(START_BATCH,END_BATCH+1):
    globals().update(_paths_for_batch(_bi))
    exec(compile(ORIGINAL_INPUT_CELL,'V7_original_manifest','exec'),globals())
    exec(compile(ORIGINAL_RESTORE_CELL,'V7_original_restore','exec'),globals())
    _manifest_by_batch[_bi]=batch_df.copy()
    if len(batch_df) != (44 if _bi==21 else 50):
        raise ValueError(f'Lô {_bi:03d} có {len(batch_df)} mẫu; cần kiểm tra lại manifest gốc')
    print('PREPARED',_bi,len(batch_df))

assert len(_manifest_by_batch)==END_BATCH-START_BATCH+1
_all_ids=[str(g) for df in _manifest_by_batch.values() for g in df.group_id]
assert len(_all_ids)==len(set(_all_ids)), 'group_id trùng giữa các batch'
assert sum(len(x) for x in _manifest_by_batch.values()) == (END_BATCH-START_BATCH+1)*50 - (6 if END_BATCH==21 else 0)

def switch_batch(bi):
    globals().update(_paths_for_batch(bi))
    globals()['batch_df']=_manifest_by_batch[bi].copy()
    for name,path in [('candidates',CANDIDATES_CSV),('v4pred',V4_PRED_CSV),
                      ('v4image',V4_IMAGE_CSV),('discovery',DISCOVERY_CSV),
                      ('feedback',FEEDBACK_CSV),('router',ROUTER_CSV)]:
        globals()[name]=read_checkpoint(path)

def stage_complete(bi, stage):
    path=_paths_for_batch(bi)[{'A':'CANDIDATES_CSV','B':'V4_IMAGE_CSV',
                               'C':'DISCOVERY_CSV','D':'FEEDBACK_CSV',
                               'E':'ROUTER_CSV','F':'FINAL_CSV'}[stage]]
    try:
        df=read_checkpoint(path)
        ids=set(_manifest_by_batch[bi].group_id.astype(str))
        if df.empty or 'group_id' not in df.columns:return False
        success=set(df.loc[df.status.astype(str).eq('ok'),'group_id'].astype(str)) if 'status' in df.columns else set(df.group_id.astype(str))
        return len(df)==len(ids) and success==ids
    except Exception:
        return False

def snapshot(bi):
    data_dir=_paths_for_batch(bi)['OUTPUT_DIR']
    if not any(data_dir.iterdir()):return None
    # tmp .zip bên ngoài thư mục gốc; os.replace giúp tránh file ZIP bị ghi dở.
    prefix=f'/kaggle/working/v7_official_test_batch_{bi:03d}_checkpoint'
    tmp_name=shutil.make_archive(prefix+'_incomplete','zip',root_dir=str(data_dir))
    final=prefix+'.zip'
    os.replace(tmp_name,final)
    return final

_orig_atomic_csv=atomic_csv
_write_counter=0

def atomic_csv(df,path):
    global _write_counter
    _orig_atomic_csv(df,path)
    _write_counter+=1
    if _write_counter % AUTO_ZIP_EVERY_WRITES==0:
        snapshot(BATCH_INDEX)

def _execute_stage(stage,bi):
    switch_batch(bi)
    if stage_complete(bi,stage):
        print(f'[SKIP] {stage} batch {bi:03d}: đã xong từ checkpoint')
        return
    print(f'\n>>> STAGE {stage} / batch {bi:03d} ({len(batch_df)} images)')
    code=globals()[f'STAGE_{stage}']
    try:
        exec(compile(code,f'V7_stage_{stage}_batch_{bi:03d}','exec'),globals())
        if not stage_complete(bi,stage):
            raise RuntimeError(f'Stage {stage} batch {bi:03d}: chưa có đủ output status=ok')
    finally:
        saved=snapshot(bi)
        print('Checkpoint:',saved)

def _unload_qwen():
    import gc,torch
    for name in ['qwen_model','qwen_processor']:
        if name in globals():
            del globals()[name]
    gc.collect();torch.cuda.empty_cache()

def _unload_countgd_a():
    import gc,torch
    global _A_COUNTGD_READY
    if 'countgd_namespace' in globals():
        countgd_namespace['model']=None
    for name in ['countgd_model','countgd_transform']:
        if name in globals():del globals()[name]
    _A_COUNTGD_READY=False
    gc.collect();torch.cuda.empty_cache()

def _unload_countgd_d():
    import gc,torch
    if 'ns' in globals(): ns['model']=None
    for name in ['model','transform']:
        if name in globals():del globals()[name]
    gc.collect();torch.cuda.empty_cache()

print(f'Đã nạp {len(_manifest_by_batch)} lô, tổng {sum(map(len,_manifest_by_batch.values()))} cặp cần chạy.')


## Môi trường GPU và CountGD++ (thực hiện một lần)

In [ ]:
# ============================================================
# CELL 2 — DEPENDENCIES + COUNTGD++ REPO
# Giữ Torch/Torchvision hiện tại.
# ============================================================

import sys
import subprocess
import importlib.metadata as im
from pathlib import Path

def pkg_version(name):
    try:
        return im.version(name)
    except Exception:
        return None

print("Existing torch:", pkg_version("torch"))
print("Existing torchvision:", pkg_version("torchvision"))
print("Existing transformers:", pkg_version("transformers"))

if not COUNTGD_REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", COUNTGD_GITHUB, str(COUNTGD_REPO)],
        check=True,
    )

req_file = COUNTGD_REPO / "requirements.txt"
if req_file.exists():
    blocked = (
        "torch", "torchvision", "torchaudio", "numpy", "scipy", "opencv",
        "gradio", "spaces", "deeplake"
    )
    keep = []
    for line in req_file.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if any(s.lower().startswith(x) for x in blocked):
            continue
        keep.append(s)

    filtered = Path("/kaggle/working/countgd_v4_requirements_filtered.txt")
    filtered.write_text("\n".join(keep) + "\n", encoding="utf-8")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered)],
        check=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "transformers>=4.57.0,<5",
        "accelerate>=1.2.0",
        "sentencepiece",
        "gdown",
        "qwen-vl-utils",
    ],
    check=True,
)

print("Dependency setup finished.")
print("transformers:", pkg_version("transformers"))


In [ ]:
# ============================================================
# CELL 3 — VERIFY TORCH/CUDA + PATCH/BUILD GROUNDINGDINO OPS
# ============================================================

import os
import re
import sys
import gc
import importlib
import subprocess
from pathlib import Path

import torch
import torchvision

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Torch CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("V3 cần GPU Kaggle.")

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

from torchvision.ops import nms
_test_boxes = torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]], device="cuda")
_test_scores = torch.tensor([0.9,0.8], device="cuda")
_ = nms(_test_boxes, _test_scores, 0.5)
print("torchvision CUDA NMS: PASS")

def detectron2_ok():
    try:
        import detectron2
        from detectron2 import _C
        return True
    except Exception as e:
        print("Detectron2 check:", repr(e))
        return False

if not detectron2_ok():
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
            "git+https://github.com/facebookresearch/detectron2.git",
        ],
        check=True,
    )
    importlib.invalidate_caches()
    if not detectron2_ok():
        raise RuntimeError("Detectron2 vẫn không load được sau khi cài.")

OPS_DIR = COUNTGD_REPO / "models" / "GroundingDINO" / "ops"
SRC_DIR = OPS_DIR / "src"
assert OPS_DIR.exists(), OPS_DIR

source_suffixes = {".h", ".hpp", ".cpp", ".cu", ".cuh"}
patched = []
for path in SRC_DIR.rglob("*"):
    if not path.is_file() or path.suffix not in source_suffixes:
        continue
    original = path.read_text(encoding="utf-8", errors="ignore")
    updated = original
    updated = re.sub(
        r"([A-Za-z_][A-Za-z0-9_]*)\.type\(\)\.is_cuda\(\)",
        r"\1.is_cuda()",
        updated,
    )
    updated = re.sub(
        r"([A-Za-z_][A-Za-z0-9_]*)\.type\(\)\.scalar_type\(\)",
        r"\1.scalar_type()",
        updated,
    )
    updated = updated.replace("value.type()", "value.scalar_type()")
    if updated != original:
        path.write_text(updated, encoding="utf-8")
        patched.append(str(path.relative_to(OPS_DIR)))

print("Patched files:", patched if patched else "already compatible")

from torch.utils.cpp_extension import CUDA_HOME
if CUDA_HOME:
    os.environ["CUDA_HOME"] = str(CUDA_HOME)

major, minor = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
os.environ["MAX_JOBS"] = "1"

if Path("/usr/bin/gcc-11").exists():
    os.environ["CC"] = "/usr/bin/gcc-11"
if Path("/usr/bin/g++-11").exists():
    os.environ["CXX"] = "/usr/bin/g++-11"

def try_import_msda():
    try:
        if str(OPS_DIR) not in sys.path:
            sys.path.insert(0, str(OPS_DIR))
        importlib.invalidate_caches()
        return importlib.import_module("MultiScaleDeformableAttention")
    except Exception:
        return None

MSDA = try_import_msda()
if MSDA is None:
    print("Building GroundingDINO CUDA ops...")
    subprocess.run(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=str(OPS_DIR),
        check=True,
    )
    importlib.invalidate_caches()
    sys.modules.pop("MultiScaleDeformableAttention", None)
    MSDA = try_import_msda()

if MSDA is None:
    raise RuntimeError("Không import được MultiScaleDeformableAttention sau build.")

print("GroundingDINO CUDA ops: PASS", MSDA.__file__)


In [ ]:
# ============================================================
# CELL 4 — DOWNLOAD COUNTGD++ WEIGHTS
# ============================================================

import sys
import subprocess
from pathlib import Path

CHECKPOINT_DIR = COUNTGD_REPO / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BERT_DIR = CHECKPOINT_DIR / "bert-base-uncased"
if not (BERT_DIR / "config.json").exists():
    print("Downloading bert-base-uncased...")
    subprocess.run(
        [sys.executable, "download_bert.py"],
        cwd=str(COUNTGD_REPO),
        check=True,
    )

COUNTGD_CKPT = CHECKPOINT_DIR / "countgd_plusplus.pth"
if (not COUNTGD_CKPT.exists()) or COUNTGD_CKPT.stat().st_size < 500 * 1024**2:
    print("Downloading CountGD++ checkpoint...")
    subprocess.run(
        [sys.executable, "-m", "gdown", COUNTGD_GDRIVE_ID, "-O", str(COUNTGD_CKPT)],
        check=True,
    )

assert (BERT_DIR / "config.json").exists()
assert COUNTGD_CKPT.exists()
print("BERT:", BERT_DIR)
print("CountGD++ checkpoint GB:", round(COUNTGD_CKPT.stat().st_size / 1024**3, 2))


In [ ]:
# CELL 7 — Optional V1 positive candidate cache; fallback = recompute CountGD.
# V1 exact cache gives closest comparability to positive branch of V4.
V1_CACHE=find_input('countgd_candidates_shard_00_of_01.csv')
if V1_CACHE is None:
    zpath=find_input('nckh_fsc147_full_count_refinement_export.zip')
    if zpath:
        with zipfile.ZipFile(zpath) as z:
            hits=[n for n in z.namelist() if n.endswith('countgd_candidates_shard_00_of_01.csv')]
            if hits:
                p=Path('/kaggle/working/v1_candidates_v7.csv')
                if not p.exists():p.write_bytes(z.read(hits[0]))
                V1_CACHE=p
if V1_CACHE:
    v1=pd.read_csv(V1_CACHE)
    v1=v1[v1['status'].astype(str).eq('ok')].copy()
    v1=v1.drop_duplicates(['image_name','target'])
    v1=v1.set_index(['image_name','target'])
    print('V1 cached candidates found:',V1_CACHE)
else:
    v1=None
    print('WARNING: V1 candidates unavailable. Will re-run CountGD++ baseline; cross-run comparability may change.')


## Các thành phần V4–V7 từ notebook đã chạy thành công

In [ ]:
# ============================================================
# CELL 11 — CROP / CONTACT-SHEET / QWEN VERIFIER UTILITIES
# ============================================================

from PIL import Image, ImageDraw, ImageFont, ImageOps
import pandas as pd
import numpy as np
import json
import re
import math
import torch

def expanded_crop(image, box, padding_frac=CROP_PADDING_FRAC):
    W,H = image.size
    x1,y1,x2,y2 = [float(v) for v in box]
    bw = max(1.0, x2-x1)
    bh = max(1.0, y2-y1)
    px = bw * padding_frac
    py = bh * padding_frac
    xx1 = int(max(0, math.floor(x1-px)))
    yy1 = int(max(0, math.floor(y1-py)))
    xx2 = int(min(W, math.ceil(x2+px)))
    yy2 = int(min(H, math.ceil(y2+py)))
    crop = image.crop((xx1,yy1,xx2,yy2)).convert("RGB")
    return crop

def make_contact_sheet(image, rows):
    n = len(rows)
    cols = 4
    rows_n = int(math.ceil(n / cols))
    label_h = 28
    sheet = Image.new(
        "RGB",
        (cols * PANEL_SIZE, rows_n * (PANEL_SIZE + label_h)),
        "white",
    )
    draw = ImageDraw.Draw(sheet)

    for local_id, row in enumerate(rows):
        box = json.loads(row["box_json"])
        crop = expanded_crop(image, box)
        crop = ImageOps.contain(crop, (PANEL_SIZE, PANEL_SIZE))

        x0 = (local_id % cols) * PANEL_SIZE
        y0 = (local_id // cols) * (PANEL_SIZE + label_h)

        panel = Image.new("RGB", (PANEL_SIZE, PANEL_SIZE), "white")
        px = (PANEL_SIZE - crop.width) // 2
        py = (PANEL_SIZE - crop.height) // 2
        panel.paste(crop, (px,py))
        sheet.paste(panel, (x0,y0))

        draw.rectangle(
            [x0, y0, x0+PANEL_SIZE-1, y0+PANEL_SIZE+label_h-1],
            outline="black",
            width=2,
        )
        draw.rectangle(
            [x0, y0+PANEL_SIZE, x0+PANEL_SIZE-1, y0+PANEL_SIZE+label_h-1],
            fill="white",
        )
        draw.text(
            (x0+6, y0+PANEL_SIZE+5),
            f"ID {local_id}",
            fill="black",
        )

    return sheet

LINE_RE = re.compile(r"^\s*(?:ID\s*)?(\d+)\s*\|\s*(YES|NO)\s*$", re.I)

def parse_sheet_output(text, n):
    parsed = {}
    for line in str(text).splitlines():
        m = LINE_RE.match(line.strip())
        if m:
            idx = int(m.group(1))
            if 0 <= idx < n:
                parsed[idx] = m.group(2).upper()
    return parsed

@torch.inference_mode()
def qwen_verify_sheet(image, target, sheet_rows):
    sheet = make_contact_sheet(image, sheet_rows)

    prompt = f"""
The second image is a numbered contact sheet of candidate detections from an object-counting model.

Target category: {target}

Task:
For EACH numbered panel, decide whether the main object shown in that panel is truly an instance of the target category "{target}".

Important:
- Judge the object in the numbered crop, not merely whether the target exists somewhere in the whole scene.
- Similar-looking or semantically related objects are NOT the target.
- Be conservative about rejecting a real target.
- Output exactly one line per panel and nothing else.
- Format exactly: 0|YES or 0|NO (use the numeric panel id; do not write the word ID)

Example:
0|YES
1|NO
2|YES
""".strip()

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "image", "image": sheet},
            {"type": "text", "text": prompt},
        ],
    }]

    text_prompt = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        image_patch_size=16,
        return_video_kwargs=True,
        return_video_metadata=True,
    )

    video_metadatas = None
    if video_inputs is not None:
        video_inputs, video_metadatas = zip(*video_inputs)
        video_inputs = list(video_inputs)
        video_metadatas = list(video_metadatas)

    inputs = qwen_processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        video_metadata=video_metadatas,
        padding=True,
        return_tensors="pt",
        do_resize=False,
        **video_kwargs,
    )
    inputs = inputs.to(next(qwen_model.parameters()).device)

    generated_ids = qwen_model.generate(
        **inputs,
        max_new_tokens=max(64, 10 * len(sheet_rows) + 32),
        do_sample=False,
    )

    trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = qwen_processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return output_text, sheet

print("Verifier utilities ready.")


In [ ]:
# ============================================================
# CELL 8 — DISCOVERY CONTACT SHEET + JSON PARSER
# ============================================================

from PIL import Image, ImageDraw, ImageOps
import json
import re
import math
import torch


def expanded_crop(image, box, padding_frac=CROP_PADDING_FRAC):
    W,H = image.size
    x1,y1,x2,y2 = [float(v) for v in box]
    bw = max(1.0, x2-x1)
    bh = max(1.0, y2-y1)
    px = bw * padding_frac
    py = bh * padding_frac
    xx1 = int(max(0, math.floor(x1-px)))
    yy1 = int(max(0, math.floor(y1-py)))
    xx2 = int(min(W, math.ceil(x2+px)))
    yy2 = int(min(H, math.ceil(y2+py)))
    return image.crop((xx1,yy1,xx2,yy2)).convert("RGB")


def make_discovery_sheet(image, rows):
    n = len(rows)
    cols = 3
    rows_n = int(math.ceil(n / cols))
    label_h = 28
    sheet = Image.new("RGB", (cols*PANEL_SIZE, rows_n*(PANEL_SIZE+label_h)), "white")
    draw = ImageDraw.Draw(sheet)

    for local_id, row in enumerate(rows):
        box = json.loads(row["box_json"]) if isinstance(row["box_json"], str) else row["box_json"]
        crop = expanded_crop(image, box)
        crop = ImageOps.contain(crop, (PANEL_SIZE, PANEL_SIZE))
        x0 = (local_id % cols) * PANEL_SIZE
        y0 = (local_id // cols) * (PANEL_SIZE + label_h)
        panel = Image.new("RGB", (PANEL_SIZE, PANEL_SIZE), "white")
        panel.paste(crop, ((PANEL_SIZE-crop.width)//2, (PANEL_SIZE-crop.height)//2))
        sheet.paste(panel, (x0,y0))
        draw.rectangle([x0,y0,x0+PANEL_SIZE-1,y0+PANEL_SIZE+label_h-1], outline="black", width=2)
        draw.text((x0+6,y0+PANEL_SIZE+5), f"ID {local_id}", fill="black")
    return sheet


def extract_json_obj(text):
    s = str(text).strip()
    s = s.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(s)
    except Exception:
        m = re.search(r"\{.*\}", s, flags=re.S)
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except Exception:
            return None


def norm_label(s):
    s = re.sub(r"[^a-z0-9 ]+", " ", str(s).lower())
    toks = [t for t in s.split() if t not in {"a","an","the","object","objects","item","items"}]
    # crude singular normalization; chỉ là safety guard, không dùng để score model.
    out=[]
    for t in toks:
        if len(t)>4 and t.endswith("ies"):
            t=t[:-3]+"y"
        elif len(t)>3 and t.endswith("s") and not t.endswith("ss"):
            t=t[:-1]
        out.append(t)
    return " ".join(out).strip()


def labels_too_close(a,b):
    a=norm_label(a); b=norm_label(b)
    if not a or not b:
        return False
    if a==b or a in b or b in a:
        return True
    A=set(a.split()); B=set(b.split())
    if not A or not B:
        return False
    j=len(A&B)/len(A|B)
    return j>=0.75


@torch.inference_mode()
def qwen_discover_confounder(image, target, rows):
    sheet = make_discovery_sheet(image, rows)

    prompt = f"""
You are helping an object-counting system identify a NON-TARGET confounder.

Target category: {target}

The second image is a numbered contact sheet of detections that a previous verifier REJECTED as not being the target.
The first image is the full scene for context.

Decide whether the rejected crops are dominated by ONE clear non-target object category that could be confusing the counting model.

Rules:
- NEVER use a synonym, subtype, supertype, plural/singular variant, or obvious paraphrase of the target as a negative category.
- If the crops are actually the target, parts of the target, mixed unrelated things, background, or uncertain: return decision NONE.
- Use decision USE only when one visually coherent non-target category clearly explains the rejected detections.
- confounder must be a short common noun phrase.
- representative_id must point to the clearest panel showing the confounder.
- Do not use any hidden labels. Judge only the images and the target text.

Return exactly one JSON object and nothing else:
{{"decision":"USE","confounder":"cashew nuts","representative_id":2}}
or
{{"decision":"NONE","confounder":"","representative_id":-1}}
""".strip()

    messages=[{
        "role":"user",
        "content":[
            {"type":"image","image":image},
            {"type":"image","image":sheet},
            {"type":"text","text":prompt},
        ],
    }]

    text_prompt=qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        image_patch_size=16,
        return_video_kwargs=True,
        return_video_metadata=True,
    )

    video_metadatas=None
    if video_inputs is not None:
        video_inputs, video_metadatas = zip(*video_inputs)
        video_inputs=list(video_inputs)
        video_metadatas=list(video_metadatas)

    inputs=qwen_processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        video_metadata=video_metadatas,
        padding=True,
        return_tensors="pt",
        do_resize=False,
        **video_kwargs,
    )
    inputs=inputs.to(next(qwen_model.parameters()).device)

    generated_ids=qwen_model.generate(**inputs, max_new_tokens=96, do_sample=False)
    trimmed=[out_ids[len(in_ids):] for in_ids,out_ids in zip(inputs.input_ids, generated_ids)]
    raw=qwen_processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    obj=extract_json_obj(raw)
    return raw,obj,sheet

print("Discovery utilities ready.")


In [ ]:
# ============================================================
# CELL 13 — COUNTGD++ FEEDBACK WRAPPER
# Three frozen branches:
# 1) baseline: reused from V4
# 2) NEG_TEXT
# 3) NEG_TEXT_PLUS_EXEMPLAR (PRIMARY)
# ============================================================

from PIL import Image
import json
import numpy as np


def parse_box_json(x):
    if isinstance(x,(list,tuple)):
        return [float(v) for v in x]
    return [float(v) for v in json.loads(str(x))]


def countgd_feedback(image_path,target,confounder=None,rep_box_xyxy=None,use_exemplar=False):
    image=Image.open(image_path).convert("RGB")
    pos_prompts={"image":image,"points":[]}

    neg_texts=[]
    neg_prompts=[]

    if confounder:
        neg_texts=[str(confounder).strip()]

    if use_exemplar and confounder and rep_box_xyxy is not None:
        x1,y1,x2,y2=[float(v) for v in rep_box_xyxy]
        neg_prompts=[{
            "image":image,
            "points":[[x1,y1,2.0,x2,y2,3.0]],
        }]

    target_clean=str(target).replace(".","").strip()
    neg_texts=[str(x).replace(".","").strip() for x in neg_texts]

    # Use the official native _predict() path but skip count()'s heatmap
    # rendering, which would add unnecessary matplotlib overhead for hundreds
    # of second-pass runs.
    boxes_norm,scores=ns["_predict"](
        image,
        target_clean,
        pos_prompts,
        neg_texts,
        neg_prompts,
    )

    boxes_norm=np.asarray(boxes_norm)
    scores_arr=np.asarray(scores)
    if boxes_norm.size:
        x0,y0,bw,bh=ns["get_box_coords_from_boxes"](image,boxes_norm)
        boxes_arr=np.stack((x0,y0,bw,bh),axis=-1)
    else:
        boxes_arr=np.empty((0,4),dtype=float)

    return {
        "count":int(len(boxes_arr)),
        "boxes_xywh":boxes_arr.tolist(),
        "scores":scores_arr.tolist() if scores_arr.size else [],
    }

print("Feedback wrapper ready.")


In [ ]:
# ============================================================
# CELL 6 — CONTACT SHEETS + ROBUST JSON PARSER
# ============================================================

from PIL import Image, ImageDraw
import json, math, re


def parse_box(x):
    if isinstance(x, (list,tuple)):
        return [float(v) for v in x]
    return [float(v) for v in json.loads(str(x))]


def expanded_crop(image, box, padding_frac=CROP_PADDING_FRAC):
    x1,y1,x2,y2 = [float(v) for v in box]
    w=max(1.0,x2-x1); h=max(1.0,y2-y1)
    px=w*padding_frac; py=h*padding_frac
    ex1=max(0,int(math.floor(x1-px)))
    ey1=max(0,int(math.floor(y1-py)))
    ex2=min(image.width,int(math.ceil(x2+px)))
    ey2=min(image.height,int(math.ceil(y2+py)))
    return image.crop((ex1,ey1,ex2,ey2))


def make_sheet(image, rows, title, max_n):
    rows = list(rows)[:max_n]
    cols=3
    n=max(1,len(rows))
    nrows=int(math.ceil(n/cols))
    panel=PANEL_SIZE
    title_h=32
    canvas=Image.new("RGB", (cols*panel, title_h+nrows*panel), "white")
    draw=ImageDraw.Draw(canvas)
    draw.text((8,8), title, fill="black")

    if len(rows)==0:
        draw.text((8,title_h+10), "NO CANDIDATES", fill="black")
        return canvas

    for local_id,row in enumerate(rows):
        box=parse_box(row["box_json"])
        crop=expanded_crop(image,box)
        crop.thumbnail((panel-12,panel-30))
        x0=(local_id%cols)*panel
        y0=title_h+(local_id//cols)*panel
        px=x0+(panel-crop.width)//2
        py=y0+22+(panel-22-crop.height)//2
        canvas.paste(crop,(px,py))
        draw.rectangle([x0,y0,x0+panel-1,y0+panel-1], outline="black", width=2)
        draw.text((x0+6,y0+3),str(local_id),fill="black")
    return canvas


def extract_json_object(text):
    s=str(text).strip()
    # strip markdown fences
    s=re.sub(r"^```(?:json)?\s*", "", s, flags=re.I)
    s=re.sub(r"\s*```$", "", s)
    # first balanced-ish object via greedy boundaries
    start=s.find("{"); end=s.rfind("}")
    if start<0 or end<start:
        return None
    frag=s[start:end+1]
    try:
        return json.loads(frag)
    except Exception:
        # conservative cleanup of single quotes only as last resort
        try:
            return json.loads(frag.replace("'", '"'))
        except Exception:
            return None

print(extract_json_object('{"target_evidence":"NONE","confounder_evidence":"HIGH","confounder":"apples"}'))


In [ ]:
# ============================================================
# CELL 8 — ROUTER FUNCTION
# Output only evidence; deterministic routing is done in Python.
# ============================================================

import torch

@torch.inference_mode()
def qwen_route_evidence(image, target, keep_sheet, reject_sheet):
    prompt=f"""
You are a safety router for an object-counting system.

Target category: {target}

Image 1: full scene.
Image 2: candidate detections that a previous verifier KEPT as the target.
Image 3: candidate detections that a previous verifier REJECTED as not the target.

Your task is NOT to count.
Judge only visual evidence.

Return these fields:
1. target_evidence:
   - STRONG: at least one clear candidate/object is truly the target.
   - WEAK: target is plausible but not visually secure.
   - NONE: no credible target evidence is visible.
2. confounder_evidence:
   - HIGH: rejected crops are dominated by one coherent non-target category.
   - MEDIUM: a likely confounder exists but evidence is mixed.
   - LOW: rejected crops are mixed/unclear.
   - NONE: no usable rejected evidence.
3. confounder: short common noun phrase, or empty string.

Important:
- Similar-looking objects are not automatically the target.
- Do not infer from hidden labels.
- Do not use the source class; it is not provided.
- If the supposed confounder is a synonym/subtype/supertype/paraphrase of the target, set confounder_evidence to LOW.
- Be conservative about declaring target_evidence NONE.

Return exactly one JSON object and nothing else:
{{"target_evidence":"STRONG|WEAK|NONE","confounder_evidence":"HIGH|MEDIUM|LOW|NONE","confounder":"..."}}
""".strip()

    messages=[{
        "role":"user",
        "content":[
            {"type":"image","image":image},
            {"type":"image","image":keep_sheet},
            {"type":"image","image":reject_sheet},
            {"type":"text","text":prompt},
        ]
    }]

    text_prompt=qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        image_patch_size=16,
        return_video_kwargs=True,
        return_video_metadata=True,
    )

    video_metadatas=None
    if video_inputs is not None:
        video_inputs, video_metadatas = zip(*video_inputs)
        video_inputs=list(video_inputs); video_metadatas=list(video_metadatas)

    inputs=qwen_processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        video_metadata=video_metadatas,
        padding=True,
        return_tensors="pt",
        do_resize=False,
        **video_kwargs,
    ).to(next(qwen_model.parameters()).device)

    generated=qwen_model.generate(**inputs,max_new_tokens=120,do_sample=False)
    trimmed=[o[len(i):] for i,o in zip(inputs.input_ids,generated)]
    raw=qwen_processor.batch_decode(trimmed,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
    return raw




In [ ]:
# ============================================================
# CELL 5 — FROZEN V7 ROUTER
# ============================================================
def route_v7(row):
    baseline = int(row["baseline_count"])
    v4 = int(row["v4_filtered_count"])
    neg_text = int(row["neg_text_count"])
    neg_ex = int(row["neg_text_exemplar_count"])

    target_evidence = str(row["target_evidence"]).upper().strip()
    v5_use = str(row["decision"]).upper().strip() == "USE"
    consensus = bool(row["target_consensus"])

    if baseline == 0:
        return "BASELINE_ZERO", 0, "baseline_already_zero"

    if target_evidence == "NONE":
        if v5_use:
            return "NEG_TEXT_EXEMPLAR", neg_ex, "target_none_and_v5_confounder_use"
        return "V4", v4, "target_none_no_reliable_confounder"

    if target_evidence == "STRONG":
        if consensus:
            return "BASELINE", baseline, "strong_target_plus_detection_consensus"
        if v5_use:
            return "NEG_TEXT", neg_text, "strong_target_but_low_detection_consensus_with_confounder"
        return "V4", v4, "strong_target_but_low_detection_consensus_no_confounder"

    if consensus:
        return "BASELINE", baseline, "weak_or_unknown_but_detection_consensus"
    if v5_use:
        return "NEG_TEXT", neg_text, "weak_or_unknown_with_confounder"
    return "V4", v4, "weak_or_unknown_fallback_v4"




In [ ]:
# Source of each inference stage is inherited from the tested per-batch notebook.
# The only performance change: keep CountGD candidate model alive within each chunk.

STAGE_A = r'''
# CELL 8 — Stage A: build boxes / baseline count; save after each image.
# V4 positive baseline original: V1 cache, filter scores strictly > 0.23.
import json, time, numpy as np
from PIL import Image

def clip_box(box,w,h):
    x1,y1,x2,y2=[float(v) for v in box]
    x1,x2=sorted((max(0,min(x1,w)),max(0,min(x2,w))))
    y1,y2=sorted((max(0,min(y1,h)),max(0,min(y2,h))))
    return [x1,y1,x2,y2]

def from_v1(row):
    key=(str(row.image_name),str(row.target))
    if v1 is None or key not in v1.index:return None
    src=v1.loc[key]
    bb=json.loads(src.boxes_json);scores=json.loads(src.scores_json)
    W,H=Image.open(row.image_path).size
    idx=sorted([i for i,s in enumerate(scores) if float(s)>COUNTGD_THRESH and i<len(bb)],key=lambda i:float(scores[i]),reverse=True)
    return {'boxes': [clip_box(bb[i],W,H) for i in idx],
            'scores':[float(scores[i]) for i in idx], 'count':len(idx), 'source':'v1_cached'}

existing=read_checkpoint(CANDIDATES_CSV)
records=existing.to_dict('records') if len(existing) else []
done=completed(existing)
needed=[r for r in batch_df.itertuples(index=False) if str(r.group_id) not in done]
recompute_any=any(from_v1(r) is None for r in needed)
countgd_loaded=bool(globals().get("_A_COUNTGD_READY", False))
if recompute_any and not countgd_loaded:
    import os,sys,gc,importlib,torch
    p=str(COUNTGD_REPO)
    if p not in sys.path:sys.path.insert(0,p)
    os.chdir(p)
    for name in list(sys.modules):
        if name=='models' or name.startswith('models.'):
            del sys.modules[name]
    importlib.invalidate_caches();gc.collect();torch.cuda.empty_cache()
    process_file=COUNTGD_REPO/'process_folder.py'
    source=process_file.read_text(encoding='utf-8')
    marker='logging.basicConfig(level=logging.INFO'
    if marker not in source:raise RuntimeError('CountGD repo structure changed: process_folder.py')
    countgd_namespace={'__name__':'v7_countgd_runtime','__file__':str(process_file)}
    exec(compile(source.split(marker,1)[0],str(process_file),'exec'),countgd_namespace)
    for k in ['build_arg_parser','build_model_and_transforms','count']:
        if k not in countgd_namespace:raise RuntimeError(f'Missing CountGD API: {k}')
    args=countgd_namespace['build_arg_parser']().parse_args([])
    args.device='cuda:0';args.pretrain_model_path=str(COUNTGD_CKPT)
    _load=torch.load
    def load_trusted(*a,**kw):kw.setdefault('weights_only',False);return _load(*a,**kw)
    torch.load=load_trusted
    try: countgd_model,countgd_transform=countgd_namespace['build_model_and_transforms'](args)
    finally: torch.load=_load
    countgd_model=countgd_model.to('cuda:0').eval()
    countgd_namespace.update(model=countgd_model,transform=countgd_transform,device=torch.device('cuda:0'))
    countgd_loaded=True
    _A_COUNTGD_READY=True

for i,r in enumerate(needed):
    t0=time.time()
    try:
        out=from_v1(r)
        if out is None:
            image=Image.open(r.image_path).convert('RGB')
            boxes,scores,_=countgd_namespace['count'](
                image,str(r.target).replace('.','').strip(),{'image':image,'points':[]},
                [],[],float(COUNTGD_THRESH),False)
            boxes=boxes.tolist() if hasattr(boxes,'tolist') else list(boxes)
            scores=scores.tolist() if hasattr(scores,'tolist') else list(scores)
            W,H=image.size
            order=sorted(range(min(len(boxes),len(scores))),key=lambda j:float(scores[j]),reverse=True)
            out={'boxes':[clip_box(boxes[j],W,H) for j in order],
                 'scores':[float(scores[j]) for j in order],
                 'count':len(order),'source':'fresh_countgd'}
        status='ok';err=''
    except Exception as e:
        out={'boxes':[],'scores':[],'count':None,'source':'error'}
        status='error';err=repr(e)
    records=[x for x in records if str(x['group_id'])!=str(r.group_id)]
    records.append({'group_id':r.group_id,'image_name':r.image_name,'target':r.target,
                    'gt_count':int(r.gt_count),'image_path':r.image_path,
                    'baseline_count':out['count'], 'boxes_json':json.dumps(out['boxes']),
                    'scores_json':json.dumps(out['scores']),'source':out['source'],
                    'status':status,'error':err,'latency_sec':round(time.time()-t0,2)})
    atomic_csv(pd.DataFrame(records),CANDIDATES_CSV)
    print('CountGD',i+1,'/',len(needed),r.group_id,status,out['count'],out['source'])

candidates=read_checkpoint(CANDIDATES_CSV)
if not candidates.empty and not candidates.status.astype(str).eq('ok').all():
    raise RuntimeError('Stage A failed for one or more rows; retry after resolving errors')
# Giữ CountGD++ trong VRAM đến khi xử lý hết phần chunk này.
print('Stage A done',len(candidates))

'''
compile(STAGE_A, 'V7_stage_A', 'exec')

STAGE_B = r'''
# CELL 11 — Stage B: Qwen3 V4 box verification + image counts (resume)
# Box cap = 80 for positive, exactly like V4.
import pandas as pd, json, time
from PIL import Image

b0=read_checkpoint(V4_IMAGE_CSV)
v4done=completed(b0)
brec=b0.to_dict('records') if len(b0) else []
p0=read_checkpoint(V4_PRED_CSV)
prec=p0.to_dict('records') if len(p0) else []
for k,row in candidates.iterrows():
    gid=str(row.group_id)
    if gid in v4done:continue
    image=Image.open(row.image_path).convert('RGB')
    boxes=json.loads(row.boxes_json)
    scores=json.loads(row.scores_json)
    verify_n=min(len(boxes),MAX_POS_BOXES_TO_VERIFY_PER_IMAGE)
    all_rows=[{'group_id':gid,'crop_id':f'{gid}__{j:04d}','image_name':row.image_name,
            'target':row.target,'image_path':row.image_path,'box_json':json.dumps(boxes[j]),
            'score':scores[j]} for j in range(verify_n)]
    row_preds=[];status='ok';error=''
    try:
        for start in range(0,len(all_rows),CONTACT_SHEET_SIZE):
            chunk=all_rows[start:start+CONTACT_SHEET_SIZE]
            raw,sheet=qwen_verify_sheet(image,row.target,chunk)
            parsed=parse_sheet_output(raw,len(chunk))
            for local_id,boxrow in enumerate(chunk):
                decision=parsed.get(local_id)
                fallback=int(decision is None)
                if fallback and KEEP_ON_PARSE_FAILURE:decision='YES'
                if decision not in ('YES','NO'):
                    raise RuntimeError('Unparseable Qwen output and fail-open disabled')
                row_preds.append({**boxrow,'vlm_pred':decision,
                    'vlm_keep':int(decision=='YES'),'parse_fallback':fallback,
                    'sheet_local_id':int(local_id),'raw_output':raw, 'status':'ok'})
    except Exception as e:
        status='error';error=repr(e)
    if status=='ok':
        # No score changes for unverified boxes; conservative keep.
        kept=sum(x['vlm_keep'] for x in row_preds)
        rejected=sum(1-x['vlm_keep'] for x in row_preds)
        filtered=int(row.baseline_count)-rejected
        prec=[x for x in prec if str(x['group_id'])!=gid]+row_preds
        atomic_csv(pd.DataFrame(prec),V4_PRED_CSV) if prec else None
        bentry={'group_id':gid,'baseline_count':int(row.baseline_count),
                'v4_filtered_count':filtered,'verified_boxes':verify_n,
                'kept_verified':kept,'unverified_kept':int(row.baseline_count)-verify_n,
                'parse_fallbacks':sum(x['parse_fallback'] for x in row_preds),
                'status':'ok','error':''}
    else:
        bentry={'group_id':gid,'baseline_count':int(row.baseline_count),
                'v4_filtered_count':None,'verified_boxes':verify_n,
                'kept_verified':None,'unverified_kept':None,'parse_fallbacks':None,
                'status':'error','error':error}
    brec=[x for x in brec if str(x['group_id'])!=gid]+[bentry]
    atomic_csv(pd.DataFrame(brec),V4_IMAGE_CSV)
    print('V4',gid,status,'filtered',bentry['v4_filtered_count'],'fallbacks',bentry['parse_fallbacks'])

v4image=read_checkpoint(V4_IMAGE_CSV)
if not v4image.status.astype(str).eq('ok').all():
    raise RuntimeError('Stage B failed; please inspect stage_b_v4_image_counts.csv')
v4pred=read_checkpoint(V4_PRED_CSV)
print('Stage B completed',len(v4image),'predicted boxes',len(v4pred))

'''
compile(STAGE_B, 'V7_stage_B', 'exec')

STAGE_C = r'''
# CELL 13 — Stage C: Qwen3 confounder discovery (resume)
# Uses only V4 rejected boxes. Never pass source_class or GT to the model.
from PIL import Image

d0=read_checkpoint(DISCOVERY_CSV)
drec=d0.to_dict('records') if len(d0) else []
ddone=completed(d0)
for i,row in candidates.iterrows():
    gid=str(row.group_id)
    if gid in ddone:continue
    if not v4pred.empty:
        rejected=v4pred[(v4pred.group_id.astype(str)==gid)&(v4pred.vlm_keep==0)].sort_values('score',ascending=False).head(DISCOVERY_REJECTED_CROPS)
    else: rejected=pd.DataFrame()
    decision='NONE';confounder='';rep_id=-1;rep_box='';raw='';status='ok';err=''
    try:
        if len(rejected):
            img=Image.open(row.image_path).convert('RGB')
            raw,obj,_=qwen_discover_confounder(img,str(row.target),rejected.to_dict('records'))
            if obj is None:raise RuntimeError('Qwen discovery JSON invalid')
            decision=str(obj.get('decision','NONE')).strip().upper()
            confounder=str(obj.get('confounder','')).strip()
            try:rep_id=int(obj.get('representative_id',-1))
            except Exception:rep_id=-1
            if decision!='USE' or not confounder or labels_too_close(confounder,row.target) or not 0<=rep_id<len(rejected):
                decision='NONE';confounder='';rep_id=-1
            else: rep_box=str(rejected.iloc[rep_id].box_json)
    except Exception as e:
        status='error';err=repr(e)
    drec=[x for x in drec if str(x['group_id'])!=gid]+[{'group_id':gid,'decision':decision,
        'confounder':confounder,'representative_id':rep_id,'representative_box_json':rep_box,
        'evidence_boxes_n':len(rejected),'raw_output':raw,'status':status,'error':err}]
    atomic_csv(pd.DataFrame(drec),DISCOVERY_CSV)
    print('V5 discovery',gid,decision,confounder,status)

discovery=read_checkpoint(DISCOVERY_CSV)
if not discovery.status.astype(str).eq('ok').all():
    raise RuntimeError('Stage C failed. Check discovery file; do not silently treat errors as NONE')

'''
compile(STAGE_C, 'V7_stage_C', 'exec')

STAGE_D = r'''
# CELL 17 — Stage D: CountGD++ negative-text / visual-exemplar feedback
f0=read_checkpoint(FEEDBACK_CSV)
frec=f0.to_dict('records') if len(f0) else []
fdone=completed(f0)
metadata=candidates.merge(discovery[['group_id','decision','confounder','representative_box_json']],on='group_id')
for i,row in metadata.iterrows():
    gid=str(row.group_id)
    if gid in fdone:continue
    baseline=int(row.baseline_count)
    nt=baseline; nex=baseline;status='ok';err=''
    try:
        if str(row.decision)=='USE' and str(row.confounder).strip():
            rep=parse_box_json(row.representative_box_json)
            nt=countgd_feedback(row.image_path,row.target,row.confounder,None,False)['count']
            nex=countgd_feedback(row.image_path,row.target,row.confounder,rep,True)['count']
    except Exception as e:
        status='error';err=repr(e)
        nt=None;nex=None
    frec=[x for x in frec if str(x['group_id'])!=gid]+[{'group_id':gid,
         'neg_text_count':nt,'neg_text_exemplar_count':nex,'status':status,'error':err}]
    atomic_csv(pd.DataFrame(frec),FEEDBACK_CSV)
    print('CountGD negative feedback',gid,nt,nex,status)
feedback=read_checkpoint(FEEDBACK_CSV)
if not feedback.status.astype(str).eq('ok').all():
    raise RuntimeError('Stage D failed; check stage_d_v5_feedback.csv')

'''
compile(STAGE_D, 'V7_stage_D', 'exec')

STAGE_E = r'''
# CELL 22 — Stage E: V6 Qwen evidence router (resume)
# This is the only V6 output V7 actually reads; V6 exemplar guard is not used in V7.
rou0=read_checkpoint(ROUTER_CSV)
rourec=rou0.to_dict('records') if len(rou0) else []
rdone=completed(rou0)
for i,row in candidates.iterrows():
    gid=str(row.group_id)
    if gid in rdone:continue
    g=v4pred[v4pred.group_id.astype(str).eq(gid)] if not v4pred.empty else pd.DataFrame()
    keep=g[g.vlm_keep==1].sort_values('score',ascending=False).head(MAX_KEEP_CROPS) if len(g) else pd.DataFrame()
    reject=g[g.vlm_keep==0].sort_values('score',ascending=False).head(MAX_REJECT_CROPS) if len(g) else pd.DataFrame()
    raw='';status='ok';err='';te='WEAK';ce='LOW';conf=''
    try:
        if int(row.baseline_count)>0:
            image=Image.open(row.image_path).convert('RGB')
            ks=make_sheet(image,keep.to_dict('records'),'V4 KEEP candidates',MAX_KEEP_CROPS)
            rs=make_sheet(image,reject.to_dict('records'),'V4 REJECT candidates',MAX_REJECT_CROPS)
            raw=qwen_route_evidence(image,str(row.target),ks,rs)
            obj=extract_json_object(raw)
            if obj is None:raise RuntimeError('Router JSON invalid')
            te=str(obj.get('target_evidence','WEAK')).upper()
            ce=str(obj.get('confounder_evidence','LOW')).upper()
            conf=str(obj.get('confounder','')).strip()
            if te not in ('STRONG','WEAK','NONE') or ce not in ('HIGH','MEDIUM','LOW','NONE'):
                raise RuntimeError('Router labels invalid: '+str(obj))
    except Exception as e:
        status='error';err=repr(e)
    rourec=[x for x in rourec if str(x['group_id'])!=gid]+[{'group_id':gid,
        'target_evidence':te,'confounder_evidence':ce,'router_confounder':conf,
        'keep_boxes_shown':len(keep),'reject_boxes_shown':len(reject),
        'raw_output':raw,'status':status,'error':err}]
    atomic_csv(pd.DataFrame(rourec),ROUTER_CSV)
    print('V6 router',gid,te,ce,status)
router=read_checkpoint(ROUTER_CSV)
if not router.status.astype(str).eq('ok').all():
    raise RuntimeError('Stage E failed; check stage_e_v6_router.csv')

'''
compile(STAGE_E, 'V7_stage_E', 'exec')

STAGE_F = r'''
# CELL 24 — Stage F: exact V7 router; write certified shard predictions
# Requires every stage to report status=ok before evaluation.
from functools import reduce
for df,path in [(candidates,CANDIDATES_CSV),(v4image,V4_IMAGE_CSV),(discovery,DISCOVERY_CSV),
                (feedback,FEEDBACK_CSV),(router,ROUTER_CSV)]:
    needed=set(batch_df.group_id.astype(str))
    got=set(df.loc[df.status.astype(str).eq('ok'),'group_id'].astype(str))
    if got!=needed:raise RuntimeError(f'Incomplete stage {path.name}: missing {sorted(needed-got)[:10]}')
merged=batch_df[['group_id','image_name','target','gt_count','split']].copy()
merged=merged.merge(candidates[['group_id','baseline_count','source']],on='group_id',validate='one_to_one')
merged=merged.merge(v4image[['group_id','v4_filtered_count','verified_boxes','parse_fallbacks']],on='group_id',validate='one_to_one')
merged=merged.merge(discovery[['group_id','decision','confounder']],on='group_id',validate='one_to_one')
merged=merged.merge(feedback[['group_id','neg_text_count','neg_text_exemplar_count']],on='group_id',validate='one_to_one')
merged=merged.merge(router[['group_id','target_evidence','confounder_evidence','keep_boxes_shown','reject_boxes_shown']],on='group_id',validate='one_to_one')
for col in ['baseline_count','v4_filtered_count','neg_text_count','neg_text_exemplar_count','keep_boxes_shown']:
    merged[col]=pd.to_numeric(merged[col],errors='raise')
merged['retention_ratio']=np.where(merged.baseline_count>0,merged.v4_filtered_count/merged.baseline_count,1.0)
merged['target_consensus']=((merged.keep_boxes_shown>=MIN_KEEP_BOXES_FOR_TARGET_CONSENSUS)
    &(merged.retention_ratio>=RETENTION_THRESHOLD)).astype(int)
run_df=merged.copy()
routes=run_df.apply(route_v7,axis=1)
run_df['v7_route']=[x[0] for x in routes]
run_df['v7_count']=[int(x[1]) for x in routes]
run_df['v7_reason']=[x[2] for x in routes]
run_df['abs_error']=(run_df.v7_count-run_df.gt_count).abs()
run_df.to_csv(AUDIT_CSV,index=False)
run_df[['group_id','v7_count']].to_csv(FINAL_CSV,index=False)
assert len(run_df)==len(batch_df) and run_df.group_id.is_unique
assert ((run_df.v7_count>=0) & (run_df.v7_count.astype(int)==run_df.v7_count)).all()
print('SUCCESS — actual inferred V7 pairs:',len(run_df), 'of',len(batch_df))
print('MAE current batch (not complete TEST):',run_df.abs_error.mean())
print('saved:',FINAL_CSV)
display(run_df[['group_id','target','gt_count','baseline_count','v4_filtered_count','decision','target_evidence','v7_route','v7_count']].head(20))

'''
compile(STAGE_F, 'V7_stage_F', 'exec')


In [ ]:
QWEN_V4_LOAD = r'''
# ============================================================
# CELL 10 — LOAD QWEN3-VL-4B
# ============================================================

import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

QWEN_DEVICE = "cuda:0"

qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
qwen_model.eval()

print("Loaded:", QWEN_MODEL_ID)
print("Model device:", next(qwen_model.parameters()).device)

'''
compile(QWEN_V4_LOAD,'QWEN_V4_LOAD','exec')

COUNTGD_D_LOAD = r'''
# ============================================================
# CELL 12 — LOAD COUNTGD++ NATIVE NEGATIVE-PROMPT API
# Dùng functions từ official test_dataset.py.
# ============================================================

import os
import sys
import torch
from pathlib import Path

if str(COUNTGD_REPO) not in sys.path:
    sys.path.insert(0,str(COUNTGD_REPO))

script_path=COUNTGD_REPO/"test_dataset.py"
source=script_path.read_text(encoding="utf-8",errors="ignore")

# Chỉ exec phần định nghĩa function, dừng trước main script.
cut=source.find("logging.basicConfig")
if cut<0:
    raise RuntimeError("Không tìm được marker logging.basicConfig trong test_dataset.py")
source_defs=source[:cut]

ns={"__file__":str(script_path),"__name__":"countgd_v5_native_runtime"}
cwd0=os.getcwd()
os.chdir(str(COUNTGD_REPO))
try:
    exec(compile(source_defs,str(script_path),"exec"),ns)
finally:
    os.chdir(cwd0)

for k in ["build_arg_parser","build_model_and_transforms","count"]:
    if k not in ns:
        raise RuntimeError(f"Missing symbol: {k}")

args=ns["build_arg_parser"]().parse_args(["--pretrain_model_path",str(COUNTGD_CKPT)])
args.device="cuda:0"

_orig_load=torch.load
def _trusted_load(*a,**kw):
    kw.setdefault("weights_only",False)
    return _orig_load(*a,**kw)
torch.load=_trusted_load

cwd0=os.getcwd()
os.chdir(str(COUNTGD_REPO))
try:
    model,transform=ns["build_model_and_transforms"](args)
finally:
    os.chdir(cwd0)
    torch.load=_orig_load

device=torch.device("cuda:0")
model=model.to(device).eval()

# Set globals used by test_dataset._predict/count.
ns["model"]=model
ns["transform"]=transform
ns["device"]=device
ns["CONF_THRESH"]=float(COUNTGD_THRESH)

print("CountGD++ native negative-prompt API loaded.")
print("CONF_THRESH:",ns["CONF_THRESH"])

'''
compile(COUNTGD_D_LOAD,'COUNTGD_D_LOAD','exec')

QWEN_E_LOAD = r'''
# ============================================================
# CELL 7 — LOAD QWEN3-VL-4B
# ============================================================

import torch, gc
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

if not torch.cuda.is_available():
    raise RuntimeError("V6 cần GPU Kaggle")

print("GPU:", torch.cuda.get_device_name(0))

qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
).eval()

print("Qwen loaded.")

'''
compile(QWEN_E_LOAD,'QWEN_E_LOAD','exec')



## Tự chạy mọi batch (1 lần Run All)

In [ ]:
# AUTO-RUN: tự chạy từng chunk (2 batch), lưu ZIP sau TỪNG stage và batch.
# Nếu đã có checkpoint ở Kaggle Input, mỗi stage tự bỏ qua phần status=ok.
import datetime,traceback
print('BẮT ĐẦU',datetime.datetime.now())

_all_batches=list(range(START_BATCH,END_BATCH+1))
_chks=[_all_batches[i:i+CHUNK_BATCHES] for i in range(0,len(_all_batches),CHUNK_BATCHES)]
for _chunk_idx,_chunk in enumerate(_chks,1):
    print(f'\n======== CHUNK {_chunk_idx}/{len(_chks)}: {_chunk} ========')
    try:
        # A: CountGD candidates; giữ model trong bộ nhớ trong chunk, không phải load 50 ảnh/lần.
        if any(not stage_complete(b,'A') for b in _chunk):
            _A_COUNTGD_READY=False
            try:
                for b in _chunk:_execute_stage('A',b)
            finally:_unload_countgd_a()

        # B + C: dùng cùng Qwen3-VL-4B và các hàm V4/V5 gốc.
        if any(not stage_complete(b,'B') or not stage_complete(b,'C') for b in _chunk):
            exec(compile(QWEN_V4_LOAD,'V7_load_qwen_v4','exec'),globals())
            try:
                for b in _chunk:
                    _execute_stage('B',b)
                    _execute_stage('C',b)
            finally:_unload_qwen()

        # D: CountGD native feedback. Chỉ init model khi có lô chưa hoàn thành.
        if any(not stage_complete(b,'D') for b in _chunk):
            exec(compile(COUNTGD_D_LOAD,'V7_load_countgd_d','exec'),globals())
            try:
                for b in _chunk:_execute_stage('D',b)
            finally:_unload_countgd_d()

        # E: Qwen evidence V6.
        if any(not stage_complete(b,'E') for b in _chunk):
            exec(compile(QWEN_E_LOAD,'V7_load_qwen_v6','exec'),globals())
            try:
                for b in _chunk:_execute_stage('E',b)
            finally:_unload_qwen()

        # F: định tuyến V7 cố định, tính count + audit.
        for b in _chunk:_execute_stage('F',b)
        print(f'CHUNK {_chunk_idx} DONE:',_chunk)
    except Exception as err:
        print('LỖI Ở CHUNK',_chunk,repr(err))
        traceback.print_exc()
        for b in _chunk:snapshot(b)
        raise

print('HOÀN THÀNH AUTO RUN',datetime.datetime.now())


## Xuất dự đoán, kiểm tra tiến độ

In [ ]:
# GỘP kết quả các batch đang có trong /kaggle/working (không bịa prediction).
import pandas as pd,zipfile,shutil
from IPython.display import FileLink,display
all_rows=[]; audit_rows=[];stages=[]
for bi in range(START_BATCH,END_BATCH+1):
    paths=_paths_for_batch(bi)
    fin=paths['FINAL_CSV'];audit=paths['AUDIT_CSV']
    ok=stage_complete(bi,'F') and audit.exists()
    stages.append({'batch_index':bi,'n_expected':len(_manifest_by_batch[bi]),
                   'n_predicted':len(pd.read_csv(fin)) if fin.exists() else 0,
                   'completed':ok})
    if ok:
        df=pd.read_csv(fin);df['batch_index']=bi;all_rows.append(df)
        au=pd.read_csv(audit);au['batch_index']=bi;audit_rows.append(au)
        snapshot(bi)
status_df=pd.DataFrame(stages)
status_df.to_csv('/kaggle/working/v7_auto_batch_status_003_021.csv',index=False)
print(status_df.to_string(index=False))
print('HOÀN THÀNH',int(status_df.completed.sum()),'/',len(status_df),'batch')
if all_rows:
    agg=pd.concat(all_rows,ignore_index=True)
    assert agg.group_id.is_unique
    agg.to_csv('/kaggle/working/v7_AUTO_new_predictions_003_021.csv',index=False)
    aud=pd.concat(audit_rows,ignore_index=True)
    aud.to_csv('/kaggle/working/v7_AUTO_new_audits_003_021.csv',index=False)
    print('Đã dự đoán:',len(agg),'/',sum(len(x) for x in _manifest_by_batch.values()))
    print('MAE của phần có kết quả:',round(aud.abs_error.mean(),4))
    display(FileLink('/kaggle/working/v7_AUTO_new_predictions_003_021.csv'))
    display(FileLink('/kaggle/working/v7_AUTO_new_audits_003_021.csv'))
print('Nếu bị ngắt phiên: Save Version/Download ZIP, Add Input checkpoint rồi Run All lần nữa.')


### Cách tiếp tục sau khi Kaggle ngắt phiên

1. Tải các ZIP `v7_official_test_batch_XXX_checkpoint.zip` trong Output (hoặc Save Version để giữ output).
2. Add Input các checkpoint từ phiên trước vào notebook.
3. **Run All** lại. Các giai đoạn đã có `status=ok` sẽ được bỏ qua. Không sửa các tham số `COUNTGD_THRESH`, `RETENTION_THRESHOLD` hoặc các prompt.

**Lưu ý:** notebook này chỉ suy luận 944 positive còn lại. Để có MAE cuối cùng của official TEST (1.301 cặp), cần gộp thêm 207 dự đoán cũ và 150 từ lô 000–002. Chỉ tính MAE cuối cùng khi đủ 1.301 cặp, không gộp số liệu ảo.
